# LeetCode 2760 — Longest Even Odd Subarray With Threshold bằng Python

Link đề: https://leetcode.com/problems/longest-even-odd-subarray-with-threshold/ · Độ khó: **Easy** · Chủ đề: Array, Sliding Window, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **mảng con liên tiếp** dài nhất thoả **ba điều kiện**: bắt đầu bằng số **chẵn**; các phần tử kề nhau **chẵn–lẻ xen kẽ**; mọi phần tử đều `<= threshold`.
2. Biết kiểm tra chẵn/lẻ bằng `x % 2`, dùng `all(...)`, cắt lát `nums[l:r + 1]`, và hiểu **mảng con (subarray)** khác **dãy con (subsequence)**.
3. Đi từ **Brute Force O(n³)** (thử mọi đoạn, kiểm tra từng đoạn) → **mở rộng tham lam O(n²)** → **quy hoạch động (DP) theo vị trí kết thúc** → **DP một biến O(1) bộ nhớ** → **hai con trỏ nhảy cóc**.
4. Rút ra **trạng thái** `cur` = độ dài đoạn hợp lệ dài nhất **kết thúc tại vị trí hiện tại** và ba trường hợp chuyển trạng thái.
5. Tránh các lỗi: quên điều kiện chẵn ở đầu, chỉ kiểm tra `threshold` ở đầu đoạn, reset `cur` về `0` thay vì `1` khi gặp số chẵn mới, và sai độ dài `r − l` / `r − l + 1`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (chỉ số từ `0`) và một số nguyên `threshold`.

Tìm **độ dài** của **mảng con (subarray) dài nhất** bắt đầu ở chỉ số `l` và kết thúc ở chỉ số `r` (`0 <= l <= r < nums.length`) thoả **tất cả** điều kiện:

1. `nums[l] % 2 == 0` (phần tử đầu là số **chẵn**).
2. Với mọi chỉ số `i` trong `[l, r - 1]`: `nums[i] % 2 != nums[i + 1] % 2` (hai phần tử kề nhau **khác tính chẵn lẻ**).
3. Với mọi chỉ số `i` trong `[l, r]`: `nums[i] <= threshold`.

Trả về độ dài lớn nhất đó. **Mảng con** là một đoạn **liên tiếp** các phần tử. Nếu không có đoạn nào thoả thì trả về `0`.

| Ví dụ | nums, threshold | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,2,5,4]`, `5` | `3` | Chọn `[2,5,4]` (`l = 1, r = 3`): `2` chẵn; `2,5,4` chẵn–lẻ–chẵn xen kẽ; tất cả `<= 5`. Không có đoạn dài hơn. |
| 2 | `[1,2]`, `2` | `1` | Chọn `[2]`: `2` chẵn, `<= 2`. Không thể lấy `[1,2]` vì đoạn phải bắt đầu bằng số chẵn. |
| 3 | `[2,3,4,5]`, `4` | `3` | Chọn `[2,3,4]`. Phần tử `5 > 4` nên không thể mở rộng thêm. |

Constraints:

- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 100`
- `1 <= threshold <= 100`

In [1]:
nums, threshold = [3, 2, 5, 4], 5
l, r = 1, 3
seg = nums[l:r + 1]
print("đoạn:", seg)
print("đầu chẵn:", seg[0] % 2 == 0)
print("xen kẽ:", all(seg[i] % 2 != seg[i + 1] % 2 for i in range(len(seg) - 1)))
print("<= threshold:", all(x <= threshold for x in seg))

đoạn: [2, 5, 4]
đầu chẵn: True
xen kẽ: True
<= threshold: True


## 3. Kiến thức nền

### 3.1. Chẵn / lẻ bằng `x % 2`

`x % 2` là số dư khi chia cho 2: `0` nếu `x` chẵn, `1` nếu `x` lẻ. Hai số **khác tính chẵn lẻ** khi `a % 2 != b % 2`.

In [2]:
for x in [2, 5, 4, 7]:
    print(x, "chẵn" if x % 2 == 0 else "lẻ", "->", x % 2)
print(2 % 2 != 5 % 2)    # True  (chẵn, lẻ)
print(4 % 2 != 2 % 2)    # False (cùng chẵn)

2 chẵn -> 0
5 lẻ -> 1
4 chẵn -> 0
7 lẻ -> 1
True
False


### 3.2. Mảng con (subarray) và cắt lát `nums[l:r + 1]`

**Mảng con** là đoạn **liên tiếp**. Dãy con (subsequence) thì được phép nhảy cóc. Với chỉ số `l..r` (gồm cả `r`), cắt lát Python là `nums[l:r + 1]` vì chỉ số kết thúc của lát cắt **không** được lấy. Độ dài của đoạn là `r − l + 1`.

In [3]:
nums = [3, 2, 5, 4]
print(nums[1:4])         # [2, 5, 4]  (l=1, r=3)
print(len(nums[1:4]), 3 - 1 + 1)   # 3 3
print(nums[0:1])         # [3]  đoạn một phần tử

[2, 5, 4]
3 3
[3]


### 3.3. `all(...)` và biểu thức sinh (generator expression)

`all(iterable)` trả `True` nếu **mọi** phần tử đúng (với iterable rỗng trả `True`). Kết hợp với biểu thức sinh `(điều_kiện for ...)` để kiểm tra "mọi phần tử đều thoả".

In [4]:
print(all([True, True, False]))                 # False
print(all(x <= 5 for x in [3, 2, 5, 4]))        # True
print(all(x <= 4 for x in [3, 2, 5, 4]))        # False
print(all(x < 0 for x in []))                   # True (rỗng)

False
True
False
True


### 3.4. `while` với nhiều điều kiện và `max`

Vòng `while a and b:` chạy khi **cả hai** điều kiện đúng; dùng `and` ngắt sớm (short-circuit): nếu `a` sai, `b` không được tính nên ta có thể viết `while r < n and nums[r] <= t:` mà không sợ lỗi chỉ số.

`best = max(best, cur)` cập nhật giá trị lớn nhất đã thấy.

In [5]:
nums = [2, 3, 4, 5]
n, r = len(nums), 0
while r < n and nums[r] <= 4:
    print("r =", r, "nums[r] =", nums[r])
    r += 1
print("dừng ở r =", r)       # 3 (vì nums[3] = 5 > 4)
best = 0
for cur in [1, 3, 2]:
    best = max(best, cur)
print(best)                   # 3

r = 0 nums[r] = 2
r = 1 nums[r] = 3
r = 2 nums[r] = 4
dừng ở r = 3
3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tự duyệt ví dụ 1: `nums = [3, 2, 5, 4]`, `threshold = 5`.

- Phần tử `3` lẻ ⇒ **không thể là đầu đoạn**.
- Phần tử `2` chẵn, `<= 5` ⇒ có thể là đầu đoạn. Đoạn bắt đầu từ vị trí 1: `2` (chẵn) → `5` (lẻ, khác tính chẵn lẻ ✓, `<= 5` ✓) → `4` (chẵn, khác `5` ✓, `<= 5` ✓) ⇒ dài `3`.

**Nhận xét quan trọng:**

1. Điều kiện **xen kẽ chẵn–lẻ** là điều kiện **giữa hai phần tử kề nhau**; điều kiện `<= threshold` là điều kiện **từng phần tử**. Cả hai đều có thể kiểm tra **cục bộ** khi mở rộng sang phải.
2. Khi đã bắt đầu đúng (đầu chẵn), mỗi bước mở rộng chỉ cần nhìn **phần tử mới** và **phần tử ngay trước nó**.
3. Nếu một đoạn hợp lệ bị gãy tại vị trí `i` (do `nums[i] > threshold` hoặc do không xen kẽ), thì mọi đoạn bắt đầu **trước `i`** và kéo dài qua `i` đều không hợp lệ.

In [6]:
nums, threshold = [3, 2, 5, 4], 5
for i, x in enumerate(nums):
    tag = []
    tag.append("chẵn" if x % 2 == 0 else "lẻ")
    tag.append("<=t" if x <= threshold else ">t")
    print(i, x, tag)

0 3 ['lẻ', '<=t']
1 2 ['chẵn', '<=t']
2 5 ['lẻ', '<=t']
3 4 ['chẵn', '<=t']


## 5. Cách 1 — Brute Force / baseline: thử mọi đoạn rồi kiểm tra từng đoạn

**Ý tưởng:** thử **mọi** cặp `(l, r)` với `0 <= l <= r < n`; với mỗi cặp viết hàm `is_valid(l, r)` kiểm tra đủ 3 điều kiện; lấy độ dài lớn nhất `r − l + 1` trong các đoạn hợp lệ.

Đây là bản **dịch thẳng đề bài**, rất dễ đúng và dùng làm **oracle** (đáp án chuẩn) để kiểm các cách sau.

In [7]:
from typing import List

class SolutionBrute:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        n = len(nums)
        best = 0

        def is_valid(l: int, r: int) -> bool:
            if nums[l] % 2 != 0:
                return False
            for i in range(l, r):
                if nums[i] % 2 == nums[i + 1] % 2:
                    return False
            for i in range(l, r + 1):
                if nums[i] > threshold:
                    return False
            return True

        for l in range(n):
            for r in range(l, n):
                if is_valid(l, r):
                    best = max(best, r - l + 1)
        return best

In [8]:
sol = SolutionBrute()
print(sol.longestAlternatingSubarray([3, 2, 5, 4], 5))      # 3
print(sol.longestAlternatingSubarray([1, 2], 2))            # 1
print(sol.longestAlternatingSubarray([2, 3, 4, 5], 4))      # 3

3
1
3


**Độ phức tạp:** có `n(n + 1)/2` đoạn, mỗi đoạn kiểm tra `O(n)` ⇒ Time `O(n³)`, Space `O(1)`. Với `n = 100` là khoảng `10^6 / 6` phép toán, vẫn nhanh, nhưng với `n` lớn thì quá chậm.

**Nút thắt:** ta **kiểm tra lại từ đầu** mỗi đoạn dù đoạn `[l, r]` chỉ hơn đoạn `[l, r − 1]` một phần tử. Nếu `[l, r − 1]` đã hợp lệ thì chỉ cần kiểm tra thêm phần tử `r`.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — mở rộng dần từ mỗi điểm đầu.** Cố định `l` (phải chẵn và `<= threshold`). Cho `r` tăng dần: đoạn `[l, r]` hợp lệ khi `[l, r − 1]` hợp lệ **và** `nums[r] <= threshold` **và** `nums[r]` khác tính chẵn lẻ với `nums[r − 1]`. Dừng ở lần đầu không thoả (vì mọi đoạn dài hơn đều chứa chỗ hỏng). Độ phức tạp giảm còn `O(n²)`.

**Đột phá 2 — trạng thái "kết thúc tại `i`".** Gọi `dp[i]` = độ dài của đoạn hợp lệ **dài nhất kết thúc tại `i`** (`0` nếu không có). Có ba trường hợp khi gặp `x = nums[i]`:

1. `x > threshold` ⇒ `dp[i] = 0` (không phần tử nào của đoạn được `> threshold`).
2. `dp[i − 1] > 0` và `x` khác tính chẵn lẻ với `nums[i − 1]` ⇒ **kéo dài**: `dp[i] = dp[i − 1] + 1`.
3. Ngược lại, nếu `x` chẵn ⇒ **bắt đầu đoạn mới**: `dp[i] = 1`; nếu `x` lẻ ⇒ `dp[i] = 0`.

Đáp án = `max(dp)` (hoặc `0` nếu mọi `dp[i] = 0`).

**Đột phá 3 — chỉ cần `dp[i − 1]`.** Giữ một biến `cur` thay cho cả mảng ⇒ Space `O(1)`.

**Đột phá 4 — nhảy cóc.** Nếu đoạn bắt đầu ở `l` kéo dài tới `r − 1` (dừng vì `nums[r]` vi phạm), thì thử `l + 1, ..., r − 1` làm điểm đầu chỉ cho đoạn **ngắn hơn** (cùng phải dừng ở `r`), nên có thể nhảy thẳng `l = r`.

In [9]:
nums, threshold = [3, 2, 5, 4, 4, 7, 2, 3], 6
cur = 0
best = 0
for i, x in enumerate(nums):
    if x > threshold:
        cur = 0
        why = "vượt threshold -> 0"
    elif cur > 0 and x % 2 != nums[i - 1] % 2:
        cur += 1
        why = "xen kẽ -> kéo dài"
    elif x % 2 == 0:
        cur = 1
        why = "số chẵn -> bắt đầu mới"
    else:
        cur = 0
        why = "số lẻ không bắt đầu -> 0"
    best = max(best, cur)
    print(f"i={i}, x={x}, cur={cur}, best={best}   ({why})")

i=0, x=3, cur=0, best=0   (số lẻ không bắt đầu -> 0)
i=1, x=2, cur=1, best=1   (số chẵn -> bắt đầu mới)
i=2, x=5, cur=2, best=2   (xen kẽ -> kéo dài)
i=3, x=4, cur=3, best=3   (xen kẽ -> kéo dài)
i=4, x=4, cur=1, best=3   (số chẵn -> bắt đầu mới)
i=5, x=7, cur=0, best=3   (vượt threshold -> 0)
i=6, x=2, cur=1, best=3   (số chẵn -> bắt đầu mới)
i=7, x=3, cur=2, best=3   (xen kẽ -> kéo dài)


## 7. Các cách tiếp theo

### Cách 2 — Với mỗi `l` hợp lệ, mở rộng `r` tham lam (O(n²))

**Từng đoạn:**

- `for l in range(n)`: chỉ xét `l` thoả `nums[l] % 2 == 0` và `nums[l] <= threshold`.
- Đặt `r = l`, rồi `while r + 1 < n and nums[r + 1] <= threshold and nums[r + 1] % 2 != nums[r] % 2: r += 1`.
- Độ dài `r − l + 1`; lấy max.

In [10]:
class SolutionExpand:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        n = len(nums)
        best = 0
        for l in range(n):
            if nums[l] % 2 != 0 or nums[l] > threshold:
                continue
            r = l
            while r + 1 < n and nums[r + 1] <= threshold and nums[r + 1] % 2 != nums[r] % 2:
                r += 1
            best = max(best, r - l + 1)
        return best

In [11]:
sol = SolutionExpand()
print(sol.longestAlternatingSubarray([3, 2, 5, 4], 5))      # 3
print(sol.longestAlternatingSubarray([1, 2], 2))            # 1
print(sol.longestAlternatingSubarray([2, 3, 4, 5], 4))      # 3

3
1
3


**Code nộp LeetCode (Cách 2):** đổi tên `SolutionExpand` thành `Solution`.

**Độ phức tạp:** mỗi `l` mở rộng tối đa `n` bước ⇒ Time `O(n²)`, Space `O(1)`.

### Cách 3 — DP theo vị trí kết thúc (mảng `dp`)

**Từng đoạn:** `dp[i]` là độ dài đoạn hợp lệ dài nhất kết thúc ở `i`. Ba trường hợp như mục 6. Đáp án `max(dp)`.

In [12]:
class SolutionDP:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        n = len(nums)
        dp = [0] * n
        for i, x in enumerate(nums):
            if x > threshold:
                dp[i] = 0
            elif i > 0 and dp[i - 1] > 0 and x % 2 != nums[i - 1] % 2:
                dp[i] = dp[i - 1] + 1
            elif x % 2 == 0:
                dp[i] = 1
            else:
                dp[i] = 0
        return max(dp)

In [13]:
sol = SolutionDP()
print(sol.longestAlternatingSubarray([3, 2, 5, 4], 5))      # 3
print(sol.longestAlternatingSubarray([1, 2], 2))            # 1
print(sol.longestAlternatingSubarray([2, 3, 4, 5], 4))      # 3
print(max([0, 0]))            # mảng dp toàn 0 -> đáp án 0

3
1
3
0


**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho `dp`. (`max(dp)` an toàn vì `n >= 1`.)

### Cách 4 — DP một biến `cur` (code nộp)

Bỏ mảng, chỉ giữ `cur = dp[i − 1]` và cập nhật `best` ngay trong vòng lặp. Đây là bản tối ưu bộ nhớ của Cách 3.

In [14]:
class SolutionRolling:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        cur = 0
        best = 0
        for i, x in enumerate(nums):
            if x > threshold:
                cur = 0
            elif cur > 0 and x % 2 != nums[i - 1] % 2:
                cur += 1
            elif x % 2 == 0:
                cur = 1
            else:
                cur = 0
            best = max(best, cur)
        return best

In [15]:
sol = SolutionRolling()
print(sol.longestAlternatingSubarray([3, 2, 5, 4], 5))      # 3
print(sol.longestAlternatingSubarray([1, 2], 2))            # 1
print(sol.longestAlternatingSubarray([2, 3, 4, 5], 4))      # 3

3
1
3


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 5 — Hai con trỏ nhảy cóc (mỗi phần tử thăm O(1) lần)

**Từng đoạn:**

- `l` duyệt; nếu `nums[l]` không là điểm đầu hợp lệ (lẻ hoặc `> threshold`) thì `l += 1`.
- Ngược lại đặt `r = l + 1` và mở rộng khi `r < n`, `nums[r] <= threshold`, `nums[r]` khác tính chẵn lẻ với `nums[r − 1]`.
- Ghi nhận `r − l` (đoạn là `l..r − 1`), rồi **nhảy** `l = r` (không phải `l + 1`).

Tại sao nhảy được: mọi điểm đầu `l'` nằm trong `(l, r)` cho đoạn dừng tại cùng chỗ `r` nên ngắn hơn. Tại `l = r` ta thử lại từ phần tử vi phạm: nó có thể chính là số chẵn hợp lệ làm điểm đầu mới.

In [16]:
class SolutionJump:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        n = len(nums)
        best = 0
        l = 0
        while l < n:
            if nums[l] % 2 != 0 or nums[l] > threshold:
                l += 1
                continue
            r = l + 1
            while r < n and nums[r] <= threshold and nums[r] % 2 != nums[r - 1] % 2:
                r += 1
            best = max(best, r - l)
            l = r
        return best

In [17]:
sol = SolutionJump()
print(sol.longestAlternatingSubarray([3, 2, 5, 4], 5))      # 3
print(sol.longestAlternatingSubarray([1, 2], 2))            # 1
print(sol.longestAlternatingSubarray([2, 3, 4, 5], 4))      # 3

3
1
3


**Độ phức tạp:** mỗi chỉ số được `l` hoặc `r` đi qua một số hằng lần ⇒ Time `O(n)`, Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 4 trên `nums = [3, 2, 5, 4, 4, 7, 2, 3]`, `threshold = 6`:

| i | `x` | chẵn? | `x <= 6`? | `cur` trước | quyết định | `cur` sau | `best` |
|---|---|---|---|---|---|---|---|
| 0 | 3 | lẻ | có | 0 | lẻ, không phải đầu | 0 | 0 |
| 1 | 2 | chẵn | có | 0 | bắt đầu đoạn mới | 1 | 1 |
| 2 | 5 | lẻ | có | 1 | khác `2` ⇒ kéo dài | 2 | 2 |
| 3 | 4 | chẵn | có | 2 | khác `5` ⇒ kéo dài | 3 | 3 |
| 4 | 4 | chẵn | có | 3 | cùng chẵn với `4` ⇒ **bắt đầu lại** | 1 | 3 |
| 5 | 7 | lẻ | **không** | 1 | vượt ngưỡng ⇒ gãy | 0 | 3 |
| 6 | 2 | chẵn | có | 0 | bắt đầu | 1 | 3 |
| 7 | 3 | lẻ | có | 1 | khác `2` ⇒ kéo dài | 2 | 3 |

Đáp án `3`. Chú ý ở `i = 4`: `4` đứng sau `4` không kéo dài được, nhưng **chính nó là số chẵn hợp lệ nên bắt đầu đoạn mới độ dài 1** (không phải `0`).

In [18]:
def trace_rolling(nums, threshold):
    cur = best = 0
    print(f"threshold = {threshold}")
    for i, x in enumerate(nums):
        before = cur
        if x > threshold:
            cur, why = 0, "vượt ngưỡng"
        elif cur > 0 and x % 2 != nums[i - 1] % 2:
            cur, why = cur + 1, "xen kẽ, kéo dài"
        elif x % 2 == 0:
            cur, why = 1, "bắt đầu đoạn mới"
        else:
            cur, why = 0, "lẻ không bắt đầu"
        best = max(best, cur)
        print(f"i={i}, x={x}, cur: {before} -> {cur}, best={best}   ({why})")
    return best

trace_rolling([3, 2, 5, 4, 4, 7, 2, 3], 6)

threshold = 6
i=0, x=3, cur: 0 -> 0, best=0   (lẻ không bắt đầu)
i=1, x=2, cur: 0 -> 1, best=1   (bắt đầu đoạn mới)
i=2, x=5, cur: 1 -> 2, best=2   (xen kẽ, kéo dài)
i=3, x=4, cur: 2 -> 3, best=3   (xen kẽ, kéo dài)
i=4, x=4, cur: 3 -> 1, best=3   (bắt đầu đoạn mới)
i=5, x=7, cur: 1 -> 0, best=3   (vượt ngưỡng)
i=6, x=2, cur: 0 -> 1, best=3   (bắt đầu đoạn mới)
i=7, x=3, cur: 1 -> 2, best=3   (xen kẽ, kéo dài)


3

In [19]:
def trace_jump(nums, threshold):
    n = len(nums)
    best = l = 0
    while l < n:
        if nums[l] % 2 != 0 or nums[l] > threshold:
            print(f"l={l}: nums[l]={nums[l]} không phải điểm đầu -> l+1")
            l += 1
            continue
        r = l + 1
        while r < n and nums[r] <= threshold and nums[r] % 2 != nums[r - 1] % 2:
            r += 1
        print(f"l={l}: mở rộng tới r={r} (đoạn {nums[l:r]}), độ dài {r - l}")
        best = max(best, r - l)
        l = r
    print("kết quả:", best)
    return best

trace_jump([3, 2, 5, 4, 4, 7, 2, 3], 6)

l=0: nums[l]=3 không phải điểm đầu -> l+1
l=1: mở rộng tới r=4 (đoạn [2, 5, 4]), độ dài 3
l=4: mở rộng tới r=5 (đoạn [4]), độ dài 1
l=5: nums[l]=7 không phải điểm đầu -> l+1
l=6: mở rộng tới r=8 (đoạn [2, 3]), độ dài 2
kết quả: 3


3

## 9. Code nộp LeetCode

Dùng Cách 4 (DP một biến), `O(n)` thời gian, `O(1)` bộ nhớ.

In [20]:
from typing import List

class Solution:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        cur = 0          # độ dài đoạn hợp lệ dài nhất KẾT THÚC tại phần tử đang xét
        best = 0
        for i, x in enumerate(nums):
            if x > threshold:                                  # vượt ngưỡng: gãy
                cur = 0
            elif cur > 0 and x % 2 != nums[i - 1] % 2:        # xen kẽ: kéo dài
                cur += 1
            elif x % 2 == 0:                                   # số chẵn: bắt đầu đoạn mới
                cur = 1
            else:                                              # số lẻ không thể làm điểm đầu
                cur = 0
            best = max(best, cur)
        return best

In [21]:
print(Solution().longestAlternatingSubarray([3, 2, 5, 4], 5))      # 3
print(Solution().longestAlternatingSubarray([1, 2], 2))            # 1
print(Solution().longestAlternatingSubarray([2, 3, 4, 5], 4))      # 3

3
1
3


## 10. Vì sao thuật toán đúng

**Định nghĩa.** `cur_i` = độ dài lớn nhất của đoạn hợp lệ **kết thúc tại `i`**, hoặc `0` nếu không có. Đáp án là `max_i cur_i`, vì mọi đoạn hợp lệ kết thúc ở một `i` nào đó.

**Quy nạp theo `i`.** Xét `x = nums[i]`.

1. Nếu `x > threshold`: không đoạn nào kết thúc ở `i` hợp lệ (điều kiện 3 đòi mọi phần tử `<= threshold`) ⇒ `cur_i = 0`. ✓.
2. Nếu `x <= threshold` và `cur_{i−1} > 0` và `x` khác tính chẵn lẻ với `nums[i − 1]`: một đoạn hợp lệ kết thúc ở `i` có độ dài `>= 2` thì bỏ phần tử cuối thu được đoạn hợp lệ kết thúc ở `i − 1`; ngược lại, thêm `x` vào đoạn hợp lệ dài nhất kết thúc ở `i − 1` cho đoạn hợp lệ (vì điều kiện giữa hai phần tử cuối được thoả, `x <= threshold`). Vậy `cur_i = cur_{i−1} + 1` là lớn nhất ✓.
3. Ngược lại (`x <= threshold` và không kéo dài được): đoạn hợp lệ kết thúc ở `i` chỉ có thể là đoạn **chỉ gồm `x`** (độ dài 1), hợp lệ khi `x` chẵn. Nếu `x` lẻ thì không có đoạn nào ⇒ `cur_i = 0` ✓.

**Quan trọng ở trường hợp 3:** ta nhánh vào đây cả khi `cur_{i−1} > 0` mà `x` **cùng** tính chẵn lẻ với phần tử trước. Khi đó `x` chẵn ⇒ `cur_i = 1` (đoạn mới), `x` lẻ ⇒ `0` (đoạn cũ bị gãy, và `x` lẻ không thể làm đầu).

**Không bỏ sót đoạn nào:** mọi đoạn hợp lệ `[l, r]` được "thấy" ở `i = r` với `cur_r >= r − l + 1` (vì `cur_r` là lớn nhất). **Không trả đoạn không hợp lệ:** `cur_i` luôn là độ dài của một đoạn hợp lệ thật (chứng minh ở các bước trên).

Cách 1 là bản dịch thẳng của đề; test ở mục 12 cho thấy mọi cách còn lại khớp với nó trên hàng nghìn input ngẫu nhiên.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force kiểm mọi đoạn | `O(n³)` | `O(1)` | `n²/2` đoạn × kiểm `O(n)` |
| 2. Mở rộng từ mỗi `l` | `O(n²)` | `O(1)` | dừng ngay khi gãy |
| 3. DP mảng | `O(n)` | `O(n)` | định nghĩa trạng thái rõ |
| 4. DP một biến | `O(n)` | `O(1)` | **code nộp** |
| 5. Hai con trỏ nhảy cóc | `O(n)` | `O(1)` | mỗi phần tử thăm O(1) lần |

Với `n <= 100` mọi cách đều chạy tức thì; điều đáng học là cách **giảm từ `O(n³)` xuống `O(n)`** bằng cách tái sử dụng kết quả của đoạn ngắn hơn.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: một phần tử chẵn / lẻ; phần tử đầu lẻ; `threshold` làm gãy đoạn; hai số chẵn liên tiếp (đoạn mới bắt đầu từ số chẵn thứ hai); toàn lẻ (đáp án `0`); mọi phần tử vượt `threshold` (đáp án `0`); mảng xen kẽ hoàn hảo.

In [22]:
sols = [SolutionBrute(), SolutionExpand(), SolutionDP(), SolutionRolling(), SolutionJump(), Solution()]
cases = [
    ("ví dụ 1", [3, 2, 5, 4], 5, 3),
    ("ví dụ 2", [1, 2], 2, 1),
    ("ví dụ 3", [2, 3, 4, 5], 4, 3),
    ("một phần tử chẵn", [2], 2, 1),
    ("một phần tử chẵn vượt ngưỡng", [4], 3, 0),
    ("một phần tử lẻ", [1], 5, 0),
    ("toàn lẻ", [1, 3, 5, 7], 100, 0),
    ("toàn vượt ngưỡng", [10, 20, 30], 5, 0),
    ("hai chẵn liên tiếp", [2, 4], 10, 1),
    ("chẵn chẵn rồi lẻ", [2, 4, 3], 10, 2),
    ("xen kẽ hoàn hảo", [2, 1, 4, 3, 6, 5], 100, 6),
    ("xen kẽ nhưng bắt đầu lẻ", [1, 2, 3, 4, 5, 6], 100, 5),
    ("ngưỡng chặn giữa", [2, 3, 8, 3, 2], 5, 2),
    ("ngưỡng bằng đúng phần tử", [2, 3, 4], 4, 3),
]
for name, nums, th, exp in cases:
    for s in sols:
        got = s.longestAlternatingSubarray(nums[:], th)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 3
PASS ví dụ 2 -> 1
PASS ví dụ 3 -> 3
PASS một phần tử chẵn -> 1
PASS một phần tử chẵn vượt ngưỡng -> 0
PASS một phần tử lẻ -> 0
PASS toàn lẻ -> 0
PASS toàn vượt ngưỡng -> 0
PASS hai chẵn liên tiếp -> 1
PASS chẵn chẵn rồi lẻ -> 2
PASS xen kẽ hoàn hảo -> 6
PASS xen kẽ nhưng bắt đầu lẻ -> 5
PASS ngưỡng chặn giữa -> 2
PASS ngưỡng bằng đúng phần tử -> 3


Giải thích vài ca:

- `[1, 2, 3, 4, 5, 6]`: đoạn không thể bắt đầu từ `1` (lẻ); bắt đầu từ `2`: `2,3,4,5,6` xen kẽ ⇒ dài `5`.
- `[2, 3, 8, 3, 2]`, `threshold = 5`: `8 > 5` chia mảng thành `[2,3]` (dài 2) và `[3,2]` (bắt đầu từ `2`: dài 1) ⇒ `2`.
- `[2, 4, 3]`: `2,4` cùng chẵn nên không nối được, nhưng `[4, 3]` hợp lệ (đầu chẵn) ⇒ dài `2`.

### 12.2. Phản ví dụ cho các lời giải sai thường gặp

Kiểm các input "bẫy": nếu lời giải reset `cur = 0` khi hai số chẵn đứng liền nhau thì `[2, 4, 3]` trả `1` thay vì `2`; nếu quên kiểm tra đầu chẵn thì `[1, 2]` trả `2` thay vì `1`; nếu chỉ kiểm ngưỡng ở đầu đoạn thì `[2, 3, 8]` với ngưỡng `5` trả `3` thay vì `2`. Mục 13 trình bày chi tiết từng lỗi.

In [23]:
tricky = [([2, 4, 3], 10, 2), ([1, 2], 2, 1), ([2, 3, 8], 5, 2)]
for nums, th, exp in tricky:
    for s in sols:
        assert s.longestAlternatingSubarray(nums[:], th) == exp
print("các input bẫy: mọi cách đúng")

các input bẫy: mọi cách đúng


### 12.3. Đối chiếu ngẫu nhiên (hợp lệ theo đề)

Sinh `n` trong `1..100`, `nums[i]` trong `1..100`, `threshold` trong `1..100`. Để có đoạn dài hơn, ta thêm các kiểu sinh có cấu trúc: dãy xen kẽ chẵn–lẻ chèn vài "điểm gãy".

In [24]:
import random
random.seed(2760)

def gen_plain():
    n = random.randint(1, 100)
    return [random.randint(1, 100) for _ in range(n)], random.randint(1, 100)

def gen_structured():
    n = random.randint(1, 100)
    nums = []
    start_even = random.random() < 0.7
    for i in range(n):
        want_even = (i % 2 == 0) == start_even
        x = random.randint(1, 50) * 2 - (0 if want_even else 1)   # chẵn 2..100 hoặc lẻ 1..99
        nums.append(x)
    for _ in range(random.randint(0, 4)):                           # chèn vài điểm gãy
        k = random.randrange(n)
        nums[k] = random.randint(1, 100)
    return nums, random.randint(30, 100)

for _ in range(400):
    for gen in (gen_plain, gen_structured):
        nums, th = gen()
        assert 1 <= len(nums) <= 100 and all(1 <= x <= 100 for x in nums) and 1 <= th <= 100
        exp = SolutionBrute().longestAlternatingSubarray(nums[:], th)
        for s in sols:
            got = s.longestAlternatingSubarray(nums[:], th)
            assert got == exp, (type(s).__name__, nums, th, got, exp)
        snap = nums[:]
        Solution().longestAlternatingSubarray(nums, th)
        assert nums == snap
print("800 ca ngẫu nhiên (400 thường + 400 có cấu trúc): 6 cách khớp, giữ nguyên dữ liệu vào")

800 ca ngẫu nhiên (400 thường + 400 có cấu trúc): 6 cách khớp, giữ nguyên dữ liệu vào


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên điều kiện "phần tử đầu phải chẵn"

In [25]:
def sai_quen_dau_chan(nums: List[int], threshold: int) -> int:
    cur = best = 0
    for i, x in enumerate(nums):
        if x > threshold:
            cur = 0
        elif cur > 0 and x % 2 != nums[i - 1] % 2:
            cur += 1
        else:
            cur = 1                    # sai: số lẻ cũng bắt đầu đoạn mới
        best = max(best, cur)
    return best

print("sai :", sai_quen_dau_chan([1, 2], 2))                    # 2
print("đúng:", Solution().longestAlternatingSubarray([1, 2], 2))  # 1

sai : 2
đúng: 1


**Vì sao sai:** đề yêu cầu `nums[l] % 2 == 0`. Đoạn `[1, 2]` xen kẽ nhưng bắt đầu bằng số lẻ nên không hợp lệ; chỉ `[2]` hợp lệ.

### Lỗi 2: Chỉ kiểm tra `threshold` ở phần tử đầu

In [26]:
def sai_chi_kiem_dau(nums: List[int], threshold: int) -> int:
    n, best = len(nums), 0
    for l in range(n):
        if nums[l] % 2 != 0 or nums[l] > threshold:      # chỉ kiểm đầu đoạn
            continue
        r = l
        while r + 1 < n and nums[r + 1] % 2 != nums[r] % 2:   # sai: quên nums[r + 1] <= threshold
            r += 1
        best = max(best, r - l + 1)
    return best

print("sai :", sai_chi_kiem_dau([2, 3, 8], 5))                    # 3
print("đúng:", Solution().longestAlternatingSubarray([2, 3, 8], 5))  # 2

sai : 3
đúng: 2


**Vì sao sai:** điều kiện `nums[i] <= threshold` áp dụng cho **mọi phần tử** của đoạn, không chỉ phần tử đầu. Phần tử `8 > 5` phải chặn đoạn.

### Lỗi 3: Reset `cur = 0` khi gặp hai số cùng tính chẵn lẻ

In [27]:
def sai_reset_ve_0(nums: List[int], threshold: int) -> int:
    cur = best = 0
    for i, x in enumerate(nums):
        if x > threshold:
            cur = 0
        elif cur > 0 and x % 2 != nums[i - 1] % 2:
            cur += 1
        elif cur == 0 and x % 2 == 0:
            cur = 1
        else:
            cur = 0                    # sai: số chẵn đứng sau số chẵn cũng phải bắt đầu đoạn mới
        best = max(best, cur)
    return best

print("sai :", sai_reset_ve_0([2, 4, 3], 10))                      # 1
print("đúng:", Solution().longestAlternatingSubarray([2, 4, 3], 10))  # 2

sai : 1
đúng: 2


**Vì sao sai:** khi `x = 4` cùng chẵn với phần tử trước (`2`), đoạn cũ gãy, nhưng `4` chẵn và `<= threshold` nên **chính nó mở đoạn mới độ dài 1** và còn có thể kéo thành `[4, 3]`. Reset về `0` làm mất đoạn đó.

### Lỗi 4: Sai độ dài `r − l` thay vì `r − l + 1`

In [28]:
def sai_do_dai(nums: List[int], threshold: int) -> int:
    n, best = len(nums), 0
    for l in range(n):
        if nums[l] % 2 != 0 or nums[l] > threshold:
            continue
        r = l
        while r + 1 < n and nums[r + 1] <= threshold and nums[r + 1] % 2 != nums[r] % 2:
            r += 1
        best = max(best, r - l)            # sai: thiếu + 1 (r là chỉ số cuối, được lấy)
    return best

print("sai :", sai_do_dai([3, 2, 5, 4], 5))                       # 2
print("đúng:", Solution().longestAlternatingSubarray([3, 2, 5, 4], 5))  # 3

sai : 2
đúng: 3


**Vì sao sai:** đoạn `l..r` có `r − l + 1` phần tử (cả hai đầu mút). Với `l = r` (một phần tử) độ dài `1`, không phải `0`.

### Lỗi 5: Trả `-1` khi không có đoạn nào

In [29]:
print(Solution().longestAlternatingSubarray([1, 3, 5], 10))   # 0, đề yêu cầu 0 (không phải -1)

0


**Lưu ý:** bài này trả `0` khi không có đoạn hợp lệ. Bài liên quan **2765 Longest Alternating Subarray** thì trả `-1`: đọc kỹ đề từng bài.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mảng con liên tiếp dài nhất thoả điều kiện **cục bộ giữa hai phần tử kề nhau**" → **DP một biến** `cur` (hoặc cửa sổ trượt) trong `O(n)`.
- Khung chuẩn: `if vi_phạm: cur = 0; elif nối_được: cur += 1; else: cur = bắt_đầu_mới_nếu_được`; `best = max(best, cur)`.
- Điều kiện kiểu "đầu đoạn phải là X" nằm ở **nhánh bắt đầu mới**, không phải nhánh nối.
- Điều kiện kiểu "mọi phần tử `<= t`" biến thành **điểm gãy**: gặp phần tử vi phạm thì `cur = 0`.
- Luôn tự hỏi: "khi đoạn gãy, phần tử hiện tại có thể mở đoạn mới không?" (bẫy lỗi 3).
- Với `n <= 100`, brute force `O(n³)` cũng chạy được; viết nó trước làm oracle rồi mới tối ưu.

## 15. Biến thể của bài

1. **Không yêu cầu đầu chẵn** (chỉ xen kẽ chẵn–lẻ và `<= threshold`): bỏ điều kiện ở nhánh bắt đầu, nhánh bắt đầu luôn `cur = 1`.
2. **Đếm số mảng con hợp lệ** (không chỉ dài nhất): với mỗi `i`, các đoạn hợp lệ kết thúc ở `i` có độ dài từ `1` tới `cur` nhưng chỉ những đoạn bắt đầu bằng số chẵn mới tính; cần đếm riêng các điểm đầu chẵn trong đoạn `cur`.
3. **Trả về chính mảng con** (hoặc chỉ số `l`, `r`): lưu thêm `start` khi `cur` đặt lại.
4. **Tổng phần tử `<= threshold`** thay vì từng phần tử: cần cửa sổ trượt thật sự.
5. **Xen kẽ theo dấu** (dương – âm), hoặc theo hiệu `+1, −1` (bài 2765).
6. **Đoạn xen kẽ dài nhất trong chuỗi nhị phân** (`0101...`): cùng khung.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | kiểm mọi `(l, r)` | `O(n³)` | `O(1)` | oracle, `n` rất nhỏ |
| 2. Mở rộng tham lam | mỗi `l`, tăng `r` tới khi gãy | `O(n²)` | `O(1)` | dễ nghĩ, `n` nhỏ |
| 3. DP mảng | `dp[i]` = dài nhất kết thúc ở `i` | `O(n)` | `O(n)` | cần xem lại mọi `dp` |
| 4. DP một biến | `cur` thay cho `dp[i − 1]` | `O(n)` | `O(1)` | **code nộp** |
| 5. Hai con trỏ nhảy cóc | `l = r` sau mỗi đoạn | `O(n)` | `O(1)` | khi cần chỉ số đầu/cuối đoạn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `7 % 2`, `8 % 2 != 3 % 2`, `all(x <= 4 for x in [2, 3, 4])`, `[5, 6, 7, 8][1:3]`.

**Bài 2 (tính tay).** Với `nums = [2, 4, 3, 6, 7, 8]`, `threshold = 7`: lập bảng `cur` và `best` từng bước theo Cách 4.

**Bài 3 (điền TODO).** Hoàn thành `longest_todo(nums, threshold)` bằng DP một biến.

In [30]:
def longest_todo(nums, threshold):
    cur = best = 0
    for i, x in enumerate(nums):
        # TODO 1: nếu x > threshold thì cur = 0
        # TODO 2: elif cur > 0 và x khác tính chẵn lẻ với nums[i - 1] thì cur += 1
        # TODO 3: elif x chẵn thì cur = 1, ngược lại cur = 0
        best = max(best, cur)
    return best

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `longest_expand(nums, threshold)` theo Cách 2 (mở rộng `r` cho mỗi `l`) mà không nhìn code mẫu. Kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `longest_alt_any_start(nums, threshold)`: bỏ điều kiện "đầu chẵn". (b) Viết `best_subarray(nums, threshold)` trả về **chính mảng con** dài nhất (nếu có nhiều đáp án, lấy cái xuất hiện đầu tiên; không có thì `[]`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `7 % 2 = 1`; `8 % 2 = 0`, `3 % 2 = 1` nên `0 != 1` là `True`; cả ba số đều `<= 4` nên `True`; `[5,6,7,8][1:3] = [6, 7]`. Bài 2: bắt đầu `cur = 0`, `x = 2` (chẵn) → `cur = 1` ...
- **Mức 2 (vừa):** Bài 3: ba nhánh đúng thứ tự: `if x > threshold: cur = 0`, `elif cur > 0 and x % 2 != nums[i-1] % 2: cur += 1`, `elif x % 2 == 0: cur = 1`, `else: cur = 0`. Bài 4: `for l in range(n)`: bỏ qua nếu `nums[l]` lẻ hoặc `> threshold`; `while` mở rộng `r`.
- **Mức 3 (sâu):** Bài 5(a): nhánh cuối luôn `cur = 1` (miễn `x <= threshold`) vì mọi phần tử đều có thể bắt đầu. Bài 5(b): theo dõi `start`: đặt `start = i` mỗi khi `cur` được gán `1`; khi `cur > best` ghi lại `(start, i)`; trả `nums[start:end + 1]`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert 7 % 2 == 1
assert (8 % 2 != 3 % 2) is True
assert all(x <= 4 for x in [2, 3, 4])
assert [5, 6, 7, 8][1:3] == [6, 7]

# Bài 2: nums = [2,4,3,6,7,8], threshold = 7
# i0: 2 -> cur=1; i1: 4 cùng chẵn -> bắt đầu lại cur=1; i2: 3 khác 4 -> cur=2;
# i3: 6 khác 3 -> cur=3; i4: 7 khác 6 -> cur=4; i5: 8 > 7 -> cur=0 ; best = 4
assert Solution().longestAlternatingSubarray([2, 4, 3, 6, 7, 8], 7) == 4

# Bài 3
def longest_todo_ans(nums, threshold):
    cur = best = 0
    for i, x in enumerate(nums):
        if x > threshold:
            cur = 0
        elif cur > 0 and x % 2 != nums[i - 1] % 2:
            cur += 1
        elif x % 2 == 0:
            cur = 1
        else:
            cur = 0
        best = max(best, cur)
    return best

assert longest_todo_ans([3, 2, 5, 4], 5) == 3
assert longest_todo_ans([1, 2], 2) == 1
assert longest_todo_ans([2, 3, 4, 5], 4) == 3

# Bài 4
def longest_expand(nums, threshold):
    n, best = len(nums), 0
    for l in range(n):
        if nums[l] % 2 != 0 or nums[l] > threshold:
            continue
        r = l
        while r + 1 < n and nums[r + 1] <= threshold and nums[r + 1] % 2 != nums[r] % 2:
            r += 1
        best = max(best, r - l + 1)
    return best

assert longest_expand([3, 2, 5, 4], 5) == 3
assert longest_expand([1, 2], 2) == 1
assert longest_expand([2, 3, 4, 5], 4) == 3

# Bài 5(a): không yêu cầu đầu chẵn
def longest_alt_any_start(nums, threshold):
    cur = best = 0
    for i, x in enumerate(nums):
        if x > threshold:
            cur = 0
        elif cur > 0 and x % 2 != nums[i - 1] % 2:
            cur += 1
        else:
            cur = 1
        best = max(best, cur)
    return best

assert longest_alt_any_start([1, 2], 2) == 2
assert longest_alt_any_start([3, 2, 5, 4], 5) == 4
assert longest_alt_any_start([1, 3, 5], 10) == 1

# Bài 5(b): trả chính mảng con
def best_subarray(nums, threshold):
    cur = best = 0
    start = 0
    best_range = (0, -1)       # rỗng: nums[0:0] = []
    for i, x in enumerate(nums):
        if x > threshold:
            cur = 0
        elif cur > 0 and x % 2 != nums[i - 1] % 2:
            cur += 1
        elif x % 2 == 0:
            cur = 1
            start = i
        else:
            cur = 0
        if cur > best:
            best = cur
            best_range = (start, i)
    return nums[best_range[0]:best_range[1] + 1]

assert best_subarray([3, 2, 5, 4], 5) == [2, 5, 4]
assert best_subarray([1, 2], 2) == [2]
assert best_subarray([1, 3, 5], 10) == []
random.seed(9)
for _ in range(300):
    nums, th = gen_structured()
    seg = best_subarray(nums, th)
    assert len(seg) == Solution().longestAlternatingSubarray(nums, th)
    if seg:
        assert seg[0] % 2 == 0 and all(x <= th for x in seg)
        assert all(seg[i] % 2 != seg[i + 1] % 2 for i in range(len(seg) - 1))
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mảng con dài nhất, điều kiện cục bộ giữa hai phần tử kề nhau)

```python
from typing import List

class Solution:
    def longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int:
        cur = best = 0
        for i, x in enumerate(nums):
            if x > threshold:
                cur = 0                                     # gãy
            elif cur > 0 and x % 2 != nums[i - 1] % 2:
                cur += 1                                    # nối
            elif x % 2 == 0:
                cur = 1                                     # bắt đầu mới (đầu phải chẵn)
            else:
                cur = 0
            best = max(best, cur)
        return best
```

**Checklist**

- [ ] Đầu đoạn là số **chẵn**; không nối thì chỉ số chẵn mới mở đoạn mới?
- [ ] Mọi phần tử `<= threshold` (không chỉ phần tử đầu)?
- [ ] Hai phần tử kề nhau **khác** tính chẵn lẻ (`!=`, không phải `==`)?
- [ ] Khi gặp số chẵn cùng tính chẵn lẻ với phần tử trước: reset về `1`, không phải `0`?
- [ ] Độ dài là `r − l + 1`; không có đoạn thì trả `0`?
- [ ] Đã thử `[1]`, `[2]`, mảng toàn lẻ, toàn vượt ngưỡng, `[2, 4, 3]`?
- [ ] Signature đúng `longestAlternatingSubarray(self, nums: List[int], threshold: int) -> int` và có `from typing import List`?